# High Stats Evaluation - Multi-File Version
This script evaluates reconstruction performance across multiple input files.
It automatically detects the number of events in each input directory.
Author: Prabhjot Singh (prabhjot@fnal.gov)
Date: 2026 March 20

In [1]:
# Case where we have multiple files with the same structure, and we want to combine the results into a single analysis.
ghosting  = True  # Set to True to include ghosting in the analysis, False to exclude ghosting
view      = "2view"  # options: "2view", "3view"
files     = "all"   #"all" # 1 for 1 file, 2 for 2 files, 3 for 3 files, etc, and all for all files in the directory
events    = "all"    # 1 for 1 event, 2 for 2 events, 3 for 3 events, etc, and all for all events in the file

# ========================================================================
# SELECTIVE FILE/EVENT FILTERING (Optional)
# ========================================================================
# Set to None to process all files/events, or specify to run only specific ones
# Example: target_file = "file2", target_event = 8  (to process only file2, event 8)
# Example: target_file = "file2", target_event = None  (to process only file2, all events)
target_file  = None   # Set to "file1", "file4", etc. to process specific file only 
target_event = None   # Set to event number (0, 1, 8, etc.) to process specific event only

# APA selection
#apa_list = ["APA0", "APA1"]  # process one or both APAs in this job; e.g. ["APA0"] for a single-APA debug run
apa_list = ["APA0"]  # process one or both APAs in this job; e.g. ["APA0"] for a single-APA debug run


In [2]:
%load_ext autoreload
%autoreload 2

# python libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
import sys
import os
from scipy.spatial import KDTree
import pandas as pd
import seaborn as sns
import time
from datetime import datetime

np.set_printoptions(linewidth=1000)

# Record job start time (used to report total job runtime at the end)
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Job started at: 2026-07-25 19:07:37


In [3]:
# Configuration: Parent directory containing multiple subdirectories
# The script will automatically find all subdirectories that contain 'xyz-coordinates'
# Each xyz-coordinates directory should contain subdirectories named with event numbers (0, 1, 2, ...)
#
# Expected structure:
# PARENT_DIR/
#   subdir1/xyz-coordinates/0/, 1/, 2/, ...
#   subdir2/xyz-coordinates/0/, 1/, 2/, ...
#   subdir3/xyz-coordinates/0/, 1/, 2/, ...

if ghosting:
    input_ghost_str = "wcp-porting-validation"
else:
    input_ghost_str = "wcp-porting-validation_without_deghosting"

#Temporary change in input directory
#PARENT_DIR = Path(f"/exp/sbnd/data/users/prabhjot/wirecell_clustering/developcode/{input_ghost_str}/sbnd/batch_results/{view}")  # CHANGE THIS to your parent directory
PARENT_DIR = Path(f"{view}")  # CHANGE THIS to your parent directory

# Number of files to process (convert 'files' variable to num_files_to_process)
num_files_to_process = None if files == "all" else files

# Number of events to process (convert 'events' variable to num_events_to_process)
num_events_to_process = None if events == "all" else events

# Output directory for plots
if ghosting:
    output_ghost_dir = "multi_file_plots_with_deghosting"
else:
    output_ghost_dir = "multi_file_plots_without_deghosting"
PLOTBASEDIR = Path(f"{output_ghost_dir}")
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

# Parameters for selection
radius_efficiency               = 2         # sce effects can cause differences in true and reco to be more than 2 cm. Once SCE is applied we need to revert this to 1 cm 
radius_purity_xz                = 2         # sce effects can cause differences in true and reco to be more than 2 cm. Once SCE is applied we need to revert this to 1 cm 
radius_purity_yz                = 5         # changing this to 5 cm from 2 cm (original) because longitudinal tracks show spread in y-directions which we already know so this should not cause purity issue
radius_purity_xy                = 5         # changing this to 5 cm from 2 cm (original) because longitudinal tracks show spread in y-directions which we already know so this should not cause purity issue
min_recopoints_threshold        = 5
min_cluster_energy              = 100
min_true_points_cutoff          = 200
min_reco_points_cutoff          = 200

ShiftReco_Z                                 = False
Draw_debugging_Plots                        = True

# Apply selections
Apply_energy_cutoff                         = True
Apply_min_true_points_cutoff                = True
Apply_min_reco_points_cutoff                = True
Apply_wire_readout_sensitive_xz_plane_cut   = True
Apply_time_window_cut                       = False
Apply_deadarea_cut                          = True

# YZ Sensitivity cut parameters
x_min = -250.0
x_max = 250.0
y_min = -200.0  #- 203.3
y_max = 200.0   # 200.5
z_min = 0.15    # 4.7
z_max = 500.85  # 500.6

# time window cuts
time_window_min = -205  #-1500 #-205 # in us
time_window_max = 1508.5 # 215   #1508.5 # in us

marker_size = 1

print("Configuration:")
print(f"Parent directory: {PARENT_DIR}")
print(f"APA(s) to process: {apa_list}")
print(f"Plot base directory: {PLOTBASEDIR}")
print(f"Time window: {time_window_min} to {time_window_max}")
print(f"Files to process: {files}")
print(f"Events to process: {events}")

# Print selective filtering info
if target_file is not None or target_event is not None:
    print(f"\n⚡ SELECTIVE FILTERING ENABLED:")
    print(f"  Target file: {target_file if target_file else 'all'}")
    print(f"  Target event: {target_event if target_event is not None else 'all'}")

# Print all cuts applied
print("\nCuts applied:")
if Apply_energy_cutoff:
    print(f"- Energy cutoff: {min_cluster_energy} MeV")
if Apply_min_true_points_cutoff:
    print(f"- Minimum true points cutoff: {min_true_points_cutoff}")
if Apply_min_reco_points_cutoff:
    print(f"- Minimum reco points cutoff: {min_reco_points_cutoff}")
if Apply_wire_readout_sensitive_xz_plane_cut:
    print(f"- Wire readout sensitive xz plane cut applied")
if Apply_time_window_cut:
    print(f"- Time window cut: {time_window_min} to {time_window_max} μs")
if Apply_deadarea_cut:
    print(f"- Dead area cut applied")

# Print cuts that are not applied
print("\nCuts not applied")
if not Apply_energy_cutoff:
    print("- Energy cutoff not applied")
if not Apply_min_true_points_cutoff:
    print("- Minimum true points cutoff not applied")
if not Apply_min_reco_points_cutoff:
    print("- Minimum reco points cutoff not applied")
if not Apply_wire_readout_sensitive_xz_plane_cut:
    print("- Wire readout sensitive xz plane cut not applied")
if not Apply_time_window_cut:
    print("- Time window cut not applied")
if not Apply_deadarea_cut:
    print("- Dead area cut not applied")

Configuration:
Parent directory: 2view
APA(s) to process: ['APA0']
Plot base directory: multi_file_plots_with_deghosting
Time window: -205 to 1508.5
Files to process: all
Events to process: all

Cuts applied:
- Energy cutoff: 100 MeV
- Minimum true points cutoff: 200
- Minimum reco points cutoff: 200
- Wire readout sensitive xz plane cut applied
- Dead area cut applied

Cuts not applied
- Time window cut not applied


In [4]:
# Import functions from Python modules
from efficiency_purity_estimate import EvaluateEfficiency, EvaluatePurity
from efficiency_purity_draw import (
    plot_efficiency_heatmap, plot_purity_heatmap,
    DrawEfficiencyVsTrueEnergyPerEvent, DrawEfficiencyVsTrueEnergyPerFile, DrawEfficiencyVsTrueEnergyPerJob,
    DrawClusterEfficiencyVsTrueEnergyPerEvent, DrawClusterEfficiencyVsTrueEnergyPerFile, DrawClusterEfficiencyVsTrueEnergyPerJob,
    DrawEfficiencyVsTrueEnergy_MatchedPairs_PerEvent, DrawEfficiencyVsTrueEnergy_MatchedPairs_PerFile, DrawEfficiencyVsTrueEnergy_MatchedPairs_PerJob,
    DrawPurityVsRecoChargePerEvent, DrawPurityVsRecoChargePerFile, DrawPurityVsRecoChargePerJob,
    DrawEfficiencyVsPurity_MatchedPairs,
    DrawAggregatedEfficiencyPlots, DrawAggregatedPurityPlots
)
from readfiles import read_files_for_event
from selections import (
    apply_energy_cutoff, apply_min_true_points_cutoff, apply_min_reco_points_cutoff,
    apply_wire_readout_sensitive_yz_plane_cut_true, apply_wire_readout_sensitive_yz_plane_cut_reco,
    reassign_cluster_ID_true, reassign_cluster_ID_reco, GroupClustersByID, ShiftRecoClusterZValues, apply_time_window_cut,
    apply_deadarea_cut_true
)
from DrawRecoTrueClusters import DrawTrueRecoClustersXZ, DrawTrueRecoClustersYZ, DrawTrueRecoClustersXY, DrawTrueClusterWithMatchedReco, DrawLabels, DrawLabelPerFile, DrawLabelPerJob, DrawTrueRecoMatchMultiplicity
from clusterpairmatching import MatchTruetoReco_OneToMany, MatchTrueToReco1to1
from bee_display_link import print_bee_display_link
from cluster_category import cluster_category
from metadata import add_metadata_true_clusters, add_metadata_true_reco_pair_cluster, add_single_metadata, aggregate_metadata, print_metadata
from process_events_to_root import new_points_accumulators, accumulate_event_points, build_event_metadata, write_root_file
from variable_pca_linearity import calculate_pca_linearity

In [5]:
def find_all_input_directories(parent_dir):
    """
    Scan parent directory for all subdirectories containing 'data' folder.
    Returns a list of file directories (file1/, file2/, etc.).
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        print(f"Error: Parent directory {parent_dir} does not exist")
        return []

    data_dirs = []
    for subdir in sorted(parent_dir.iterdir()):
        if subdir.is_dir():
            data_path = subdir / "data"
            if data_path.exists() and data_path.is_dir():
                data_dirs.append(subdir)
                print(f"Found: {subdir}")

    return data_dirs

def detect_events_in_directory(input_dir):
    """
    Auto-detect the number of events in a directory.
    Events are identified as numeric subdirectories in data/.
    Returns a sorted list of event numbers.
    """
    input_dir = Path(input_dir)
    data_dir = input_dir / "data"
    
    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return []
    
    events = []
    for item in data_dir.iterdir():
        if item.is_dir():
            try:
                event_num = int(item.name)
                events.append(event_num)
            except ValueError:
                pass
    
    return sorted(events)

# Auto-detect all input directories from parent directory
print(f"Scanning parent directory: {PARENT_DIR}")
print("-" * 60)
input_directories = find_all_input_directories(PARENT_DIR)
# Limit to num_files_to_process files for testing
if num_files_to_process is not None:
    input_directories = input_directories[:num_files_to_process]
else:
    num_files_to_process = len(input_directories)
print("-" * 60)

print(f"\nFound {len(input_directories)} input directories with data/\n")
if input_directories:
    for input_dir in input_directories:
        events = detect_events_in_directory(input_dir)
        if events:
            print(f"  {input_dir.name}/data/: {len(events)} events ({min(events)}-{max(events)})")
        else:
            print(f"  {input_dir.name}/data/: No events found")
else:
    print(f"Error: No subdirectories with 'data/' found in {PARENT_DIR}")
    print("    Please check that:")
    print("    1. PARENT_DIR is set correctly")
    print("    2. Subdirectories (file1/, file2/, etc.) contain 'data/' folders")
    print("    3. data/ folders contain event directories (0, 1, 2, ...)")

Scanning parent directory: 2view
------------------------------------------------------------
Found: 2view/file1
Found: 2view/file10
Found: 2view/file2
Found: 2view/file3
Found: 2view/file4
Found: 2view/file5
Found: 2view/file6
Found: 2view/file7
Found: 2view/file8
Found: 2view/file9
------------------------------------------------------------

Found 10 input directories with data/

  file1/data/: 8 events (0-7)
  file10/data/: 15 events (0-14)
  file2/data/: 9 events (0-8)
  file3/data/: 11 events (0-10)
  file4/data/: 14 events (0-13)
  file5/data/: 15 events (0-14)
  file6/data/: 11 events (0-10)
  file7/data/: 11 events (0-10)
  file8/data/: 13 events (0-12)
  file9/data/: 11 events (0-10)


In [6]:
# ============================================================================
# RESTRUCTURED MAIN PROCESSING LOOP - HIERARCHICAL EVENT/FILE/JOB ANALYSIS
# ============================================================================
# Structure:
#   Job Level
#     ├─ File Level Loop
#     │   ├─ Event Level Loop
#     │   │   ├─ Efficiency/Purity Heatmaps (all clusters, no pairing)
#     │   │   ├─ True Cluster Visualizations (with matched reco clusters)
#     │   │   ├─ Efficiency vs True Energy plots (per true cluster)
#     │   │   └─ Purity vs Reco Charge plots (per reco cluster)
#     │   ├─ Event-Level Aggregation (2D/1D plots for all clusters in event)
#     │   └─ File-Level Aggregation (2D/1D plots for all clusters in file)
#     └─ Job-Level Aggregation (2D/1D plots across all files)
# ============================================================================

from datetime import datetime, timedelta
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")  # shared across both APA iterations in this job

for apa in apa_list:
    print(f"\n{'#'*70}")
    print(f"PROCESSING {apa}")
    print(f"{'#'*70}\n")

    if not input_directories:
        print("No input directories found. Cannot proceed with processing.")
    else:
        
        output_dir_view = PLOTBASEDIR / f"{view}"
        output_dir      = output_dir_view / f"apa_{apa}_{timestamp}"
        
        # Create output directory
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"\n{'='*70}")
        print(f"📁 Output directory created with timestamp:")
        print(f"   {output_dir}")
        print(f"{'='*70}\n")
        
        # Aggregate containers for each level
        job_efficiency_results  = []      # All efficiency results across all files/events
        job_purity_results      = []      # All purity results across all files/events
        job_matched_pairs       = []      # Matched pairs across all files/events
        input_directories_map   = {}      # Track event -> (input_dir, evt_num) mapping
        job_bee_links           = []      # Collect bee display links for all files
        job_metadata_list       = []      # Metadata for all true clusters across all files/events
        job_pair_metadata_list  = []      # Metadata for 1-to-1 true-reco pairs across all files/events
        root_true_points_cols, root_reco_points_cols, root_true_points_before_deadarea_cols = new_points_accumulators()  # Phase A ROOT output (per APA, written into this APA's job_summary)
        root_true_cluster_metadata_list = []      # Phase A ROOT output: true cluster metadata across all files/events
        root_reco_cluster_metadata_list = []      # Phase A ROOT output: reco cluster metadata across all files/events
        root_pair_metadata_list         = []      # Phase A ROOT output: 1-to-1 true-reco pair metadata across all files/events
        
        total_events_processed  = 0
        total_files_processed   = 0

        # ========================================================================
        # FILE LEVEL LOOP
        # ========================================================================
        for file_idx, input_dir in enumerate(input_directories):
            input_file_name = input_dir.name
            
            # SELECTIVE FILTERING: Skip files that don't match target_file
            if target_file is not None and input_file_name != target_file:
                print(f"Skipping {input_file_name} (target: {target_file})")
                continue
            
            print(f"\n{'='*70}")
            # in following also print exact name of the input directory being processed
            print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
            
            print(f"{'='*70}")
            
            file_output_dir = output_dir / input_file_name
            file_output_dir.mkdir(parents=True, exist_ok=True)
            
            # Containers for file-level aggregation
            file_efficiency_results = []
            file_purity_results     = []
            file_matched_pairs      = []
            file_metadata_list      = []      # Metadata for all true clusters in this file
            file_pair_metadata_list = []      # Metadata for 1-to-1 true-reco pairs in this file
            
            # Get event range for this directory
            events_list = detect_events_in_directory(input_dir)
            if not events_list:
                print(f"No events found in {input_dir}, skipping...")
                continue
            
            event_low  = min(events_list)
            # Determine event_high based on num_events_to_process
            if num_events_to_process is None:
                event_high = max(events_list) + 1  # Process all events
            else:
                event_high = event_low + num_events_to_process  # Process specified number of events

            bee_url = print_bee_display_link(input_dir)
            if bee_url:
                job_bee_links.append({'file': input_file_name, 'url': bee_url})
            print(f"Processing events {event_low} to {event_high-1}\n")
            
            # ====================================================================
            # EVENT LEVEL LOOP
            # ====================================================================
            for evt in range(event_low, event_high):
                # SELECTIVE FILTERING: Skip events that don't match target_event
                if target_event is not None and evt != target_event:
                    continue
                
                print(f"  {'='*60}")
                print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
                print(f"  EVENT {evt}")
                print(f"  {'='*60}")
                
                result = read_files_for_event(input_dir, evt, apa)
                if result is None:
                    print(f"  Could not read data, skipping event {evt}")
                    continue
                
                # Setup event output directories
                event_key                        = f"{input_file_name}_{evt}"
                input_directories_map[event_key] = (input_dir, evt)
                
                event_output_dir = file_output_dir / f"event_{evt:03d}"
                event_output_dir.mkdir(parents=True, exist_ok=True)
                PLOTDIR_EVT     = event_output_dir
                
                efficiency_dir  = event_output_dir / "efficiency"
                purity_dir      = event_output_dir / "purity"
                efficiency_dir  .mkdir(parents=True, exist_ok=True)
                purity_dir      .mkdir(parents=True, exist_ok=True)

                # Sub-directories for efficiency-vs-purity matched-pair plots: one including
                # unmatched true clusters (drawn in a dedicated "no match" box), one excluding them
                eff_vs_purity_incl_dir = event_output_dir / "true_reco_matched_pair_efficiency_purity_including_unmatched_true_clusters"
                eff_vs_purity_excl_dir = event_output_dir / "true_reco_matched_pair_efficiency_purity_excluding_unmatched_true_clusters"
                eff_vs_purity_incl_dir.mkdir(parents=True, exist_ok=True)
                eff_vs_purity_excl_dir.mkdir(parents=True, exist_ok=True)

                # Sub-directories for 2D/1D efficiency-vs-true-energy plots, split by matching level
                efficiency_2d1d_imaging_dir              = efficiency_dir / "efficiency_2d_1d_imaging_level"
                efficiency_2d1d_clustering_dir           = efficiency_dir / "efficiency_2d_1d_clusteringlevel"
                efficiency_2d1d_clustering_pairs_only_dir = efficiency_dir / "efficiency_2d_1d_clusteringlevel_true_reco_pairs_only"
                efficiency_2d1d_imaging_dir              .mkdir(parents=True, exist_ok=True)
                efficiency_2d1d_clustering_dir           .mkdir(parents=True, exist_ok=True)
                efficiency_2d1d_clustering_pairs_only_dir.mkdir(parents=True, exist_ok=True)
                
                # Load and process data
                x_true, y_true, z_true, id_true, q_true, e_true, t_true, x_pred, y_pred, z_pred, id_pred, q_pred = result
                
                true_5d_points              = np.column_stack((x_true, y_true, z_true, id_true, q_true, e_true, t_true))
                true_5d_points              = reassign_cluster_ID_true(true_5d_points)
                true_5d_points_unfiltered   = true_5d_points.copy()  # Keep an unfiltered copy for later comparison

                # Apply selections to true and reco points
                # True points here
                if Apply_energy_cutoff:
                    true_5d_points = apply_energy_cutoff(true_5d_points, min_cluster_energy)
                if Apply_min_true_points_cutoff:
                    true_5d_points = apply_min_true_points_cutoff(true_5d_points, min_true_points_cutoff)
                if Apply_wire_readout_sensitive_xz_plane_cut:
                    true_5d_points = apply_wire_readout_sensitive_yz_plane_cut_true(true_5d_points, x_min, x_max, y_min, y_max, z_min, z_max)
                deadarea_info = {}
                if Apply_deadarea_cut:
                    pre_cids, pre_counts = np.unique(true_5d_points[:, 3], return_counts=True)
                    before_counts = dict(zip(pre_cids, pre_counts))
                    pre_cut_points = {c: true_5d_points[true_5d_points[:, 3] == c] for c in pre_cids}
                    true_5d_points = apply_deadarea_cut_true(true_5d_points, apa, view_type=view, output_dir=PLOTDIR_EVT, event=evt, file_name=input_file_name)
                    if len(true_5d_points) > 0:
                        post_cids, post_counts = np.unique(true_5d_points[:, 3], return_counts=True)
                        after_counts = dict(zip(post_cids, post_counts))
                    else:
                        after_counts = {}
                    deadarea_info = {"before": before_counts, "after": after_counts, "pre_cut_points": pre_cut_points}
                if Apply_time_window_cut:
                    true_5d_points = apply_time_window_cut(true_5d_points, time_window_min, time_window_max, apa)

                # Reco points here
                clusters_true            = GroupClustersByID(true_5d_points)
                clusters_true_unfiltered = GroupClustersByID(true_5d_points_unfiltered)
                
                predicted_5d_points = np.column_stack((x_pred, y_pred, z_pred, id_pred, q_pred))
                if Apply_min_reco_points_cutoff:
                    predicted_5d_points = apply_min_reco_points_cutoff(predicted_5d_points, min_reco_points_cutoff)
                if Apply_wire_readout_sensitive_xz_plane_cut:
                    predicted_5d_points = apply_wire_readout_sensitive_yz_plane_cut_reco(predicted_5d_points, x_min, x_max, y_min, y_max, z_min, z_max)

                # All selections applied above

                # Reassign cluster IDs and group into clusters after all selections are applied
                predicted_5d_points = reassign_cluster_ID_reco(predicted_5d_points)
                clusters_reco       = GroupClustersByID(predicted_5d_points)
                
                if ShiftReco_Z:
                    clusters_reco = ShiftRecoClusterZValues(clusters_reco, shift_value=0.5)

                # Call function to draw the true and the reco clusters together in the same plot to visually compare them
                # We will use different colors for true and reco clusters.
                from DrawRecoTrueClusters import DrawTrueRecoClustersXZ, DrawTrueRecoClustersYZ, DrawTrueRecoClustersXY
                DrawTrueRecoClustersXZ(clusters_true, clusters_reco, evt, apa, PLOTDIR_EVT, input_file_name)
                DrawTrueRecoClustersYZ(clusters_true, clusters_reco, evt, apa, PLOTDIR_EVT, input_file_name)
                DrawTrueRecoClustersXY(clusters_true, clusters_reco, evt, apa, PLOTDIR_EVT, input_file_name)
                DrawLabels(clusters_true, evt, apa, PLOTDIR_EVT, input_file_name)

                # Analyze cluster categories based on angle in XZ plane
                print(f"    Analyzing cluster categories for {len(clusters_true)} true clusters...")
                cluster_category_results = cluster_category(clusters_true, output_dir=PLOTDIR_EVT, event=evt, apa=apa, file_name=input_file_name)
                print(f"    Cluster category analysis complete: {len(cluster_category_results)} clusters analyzed")

                # ================================================================
                # EVALUATE EFFICIENCY AND PURITY
                # ================================================================
                from efficiency_purity_estimate import EvaluateEfficiency, EvaluatePurity
                efficiency_results  = EvaluateEfficiency(clusters_true, clusters_reco, event_key, radius_efficiency, min_recopoints_threshold)
                job_efficiency_results.extend(efficiency_results)

                purity_results      = EvaluatePurity(clusters_true, clusters_reco, event_key, radius_purity_xz, radius_purity_yz, radius_purity_xy)
                job_purity_results.extend(purity_results)

                # ================================================================
                # PHASE A ROOT OUTPUT (accumulate this event's points + metadata;
                # written once per APA into that APA's job_summary directory below)
                # ================================================================
                accumulate_event_points(root_true_points_cols, root_reco_points_cols, root_true_points_before_deadarea_cols,
                                        input_file_name, evt, apa, clusters_true, clusters_reco, deadarea_info)
                root_event_true_metadata, root_event_reco_metadata, root_event_pair_metadata = build_event_metadata(
                    input_file_name, evt, apa, view, event_key,
                    clusters_true, clusters_reco, cluster_category_results,
                    efficiency_results, purity_results, deadarea_info)
                root_true_cluster_metadata_list.extend(root_event_true_metadata)
                root_reco_cluster_metadata_list.extend(root_event_reco_metadata)
                root_pair_metadata_list.extend(root_event_pair_metadata)
                
                # ================================================================
                # COLLECT CLUSTER METADATA
                # ================================================================
                print(f"    Collecting cluster metadata for {len(clusters_true)} true clusters...")
                event_metadata_list = add_metadata_true_clusters(
                    efficiency_results,
                    cluster_category_results,
                    file_name=input_file_name,
                    event=evt,
                    apa=apa,
                    view=view,
                    event_key=event_key
                )
                print(f"    Metadata collected for {len(event_metadata_list)} clusters")

                # Compute PCA linearity per true cluster and attach it to the event metadata
                linearity_lookup = {
                    (input_file_name, event_key, apa, cluster_id): calculate_pca_linearity(cluster_points)
                    for cluster_id, cluster_points in clusters_true.items()
                }
                add_single_metadata(event_metadata_list, 'linearity', linearity_lookup)

                # Display event-level metadata
                if target_file != None and target_event != None:
                    Print_Metadata = True  # Set to True to print event-level metadata summary
                else:
                    Print_Metadata = False
                if Print_Metadata and event_metadata_list:
                    print_metadata(event_metadata_list)
                
                # Aggregate to file and job levels
                file_metadata_list.extend(event_metadata_list)
                job_metadata_list.extend(event_metadata_list)

                # Bar chart: how many true clusters match to how many reco clusters (event level)
                DrawTrueRecoMatchMultiplicity(event_metadata_list, event_output_dir, apa, f'Event {evt}', f'event_{evt}', file_name=input_file_name)
                
                # ================================================================
                # COLLECT 1-TO-1 TRUE-RECO PAIR METADATA
                # ================================================================
                event_matched_pairs = MatchTrueToReco1to1(efficiency_results, purity_results)
                event_pair_metadata_list = add_metadata_true_reco_pair_cluster(
                    event_matched_pairs,
                    cluster_category_results,
                    file_name=input_file_name,
                    event=evt,
                    apa=apa,
                    view=view,
                    event_key=event_key
                )
                print(f"    Pair metadata collected for {len(event_pair_metadata_list)} true-reco pairs")
                
                # ================================================================
                # EVENT-LEVEL PROCESSING: HEATMAPS AND TRUE CLUSTER VISUALIZATIONS
                # ================================================================
                print(f"    [1/5] Drawing efficiency/purity heatmaps...")
                print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
                print(f"  EVENT {evt}")
                from efficiency_purity_draw import plot_efficiency_heatmap, plot_purity_heatmap
                plot_efficiency_heatmap(efficiency_results, evt, apa, efficiency_dir, input_file_name)
                plot_purity_heatmap(purity_results, evt, apa, purity_dir, input_file_name)
                
                print(f"    [2/5] Drawing true clusters with matched reco clusters...")
                print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
                print(f"  EVENT {evt}")
                matched_true_reco_clusters = MatchTruetoReco_OneToMany(purity_results, efficiency_results)
                for matched_info in matched_true_reco_clusters:
                    DrawTrueClusterWithMatchedReco(matched_info, clusters_true, clusters_reco, efficiency_dir, evt, apa, input_file_name)
                
                print(f"    [3/5] Drawing efficiency vs true cluster energy plots...")
                print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
                print(f"  EVENT {evt}")
                DrawEfficiencyVsTrueEnergyPerEvent(efficiency_results, efficiency_2d1d_imaging_dir, evt, apa, input_file_name, cluster_category_results=cluster_category_results)
                DrawClusterEfficiencyVsTrueEnergyPerEvent(event_pair_metadata_list, efficiency_2d1d_clustering_dir, evt, apa, input_file_name, all_true_metadata_list=event_metadata_list)
                DrawEfficiencyVsTrueEnergy_MatchedPairs_PerEvent(event_pair_metadata_list, efficiency_2d1d_clustering_pairs_only_dir, evt, apa, input_file_name)


                print(f"    [4/5] Drawing purity vs reco charge plots...")
                print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
                print(f"  EVENT {evt}")
                DrawPurityVsRecoChargePerEvent(event_pair_metadata_list, purity_dir, evt, apa, input_file_name)
                DrawEfficiencyVsPurity_MatchedPairs(event_pair_metadata_list, eff_vs_purity_incl_dir, f'Event {evt}', apa, input_file_name, all_true_metadata_list=event_metadata_list)
                DrawEfficiencyVsPurity_MatchedPairs(event_pair_metadata_list, eff_vs_purity_excl_dir, f'Event {evt}', apa, input_file_name, all_true_metadata_list=None)
                
                # Aggregate results
                file_efficiency_results.    extend(efficiency_results)
                file_purity_results.        extend(purity_results)
                file_pair_metadata_list.    extend(event_pair_metadata_list)
                job_pair_metadata_list.     extend(event_pair_metadata_list)

                total_events_processed += 1
                print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
                print(f"  EVENT {evt}")
                print(f"    [5/5] Event {evt} complete: {len(clusters_true)} true, {len(clusters_reco)} reco clusters\n")
            
            # ====================================================================
            # FILE-LEVEL AGGREGATION (After all events in file are processed)
            # ====================================================================
            print(f"\n{'='*70}")
            print(f"FILE-LEVEL AGGREGATION: Generating file-level summary plots...")
            print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
            print(f"{'='*70}")
            print(f"Total events processed in file: {total_events_processed}")
            print(f"Total efficiency results in file: {len(file_efficiency_results)}")
            print(f"Total purity results in file: {len(file_purity_results)}")
            print(f"Total matched pairs in file: {len(file_matched_pairs)}")
            print(f"Total 1-to-1 true-reco pairs in file: {len(file_pair_metadata_list)}")
            if file_efficiency_results and file_purity_results:
                print(f"\n  FILE-LEVEL AGGREGATION: Generating file-level summary plots...")
                print(f"  Total clusters in file: {len(file_efficiency_results)} efficiency, {len(file_purity_results)} purity")
                
                agg_output_dir = file_output_dir / "file_summary"
                agg_output_dir.mkdir(parents=True, exist_ok=True)

                # Sub-directories for 2D/1D efficiency-vs-true-energy plots, split by matching level
                file_efficiency_2d1d_imaging_dir              = agg_output_dir / "efficiency" / "efficiency_2d_1d_imaging_level"
                file_efficiency_2d1d_clustering_dir           = agg_output_dir / "efficiency" / "efficiency_2d_1d_clusteringlevel"
                file_efficiency_2d1d_clustering_pairs_only_dir = agg_output_dir / "efficiency" / "efficiency_2d_1d_clusteringlevel_true_reco_pairs_only"
                file_efficiency_2d1d_imaging_dir              .mkdir(parents=True, exist_ok=True)
                file_efficiency_2d1d_clustering_dir           .mkdir(parents=True, exist_ok=True)
                file_efficiency_2d1d_clustering_pairs_only_dir.mkdir(parents=True, exist_ok=True)

                # Sub-directory for purity-vs-reco-charge plots
                file_purity_summary_dir = agg_output_dir / "purity"
                file_purity_summary_dir.mkdir(parents=True, exist_ok=True)

                # Sub-directories for efficiency-vs-purity matched-pair plots: one including
                # unmatched true clusters (drawn in a dedicated "no match" box), one excluding them
                file_eff_vs_purity_incl_dir = agg_output_dir / "true_reco_matched_pair_efficiency_purity_including_unmatched_true_clusters"
                file_eff_vs_purity_excl_dir = agg_output_dir / "true_reco_matched_pair_efficiency_purity_excluding_unmatched_true_clusters"
                file_eff_vs_purity_incl_dir.mkdir(parents=True, exist_ok=True)
                file_eff_vs_purity_excl_dir.mkdir(parents=True, exist_ok=True)

                DrawEfficiencyVsTrueEnergyPerFile(file_efficiency_results, file_efficiency_2d1d_imaging_dir, apa, input_file_name, file_metadata_list=file_metadata_list)
                DrawClusterEfficiencyVsTrueEnergyPerFile(file_pair_metadata_list, file_efficiency_2d1d_clustering_dir, apa, input_file_name, all_true_metadata_list=file_metadata_list)
                DrawEfficiencyVsTrueEnergy_MatchedPairs_PerFile(file_pair_metadata_list, file_efficiency_2d1d_clustering_pairs_only_dir, apa, input_file_name)
                DrawPurityVsRecoChargePerFile(file_pair_metadata_list, file_purity_summary_dir, apa, input_file_name)
                DrawEfficiencyVsPurity_MatchedPairs(file_pair_metadata_list, file_eff_vs_purity_incl_dir, 'File Level', apa, input_file_name, all_true_metadata_list=file_metadata_list)
                DrawEfficiencyVsPurity_MatchedPairs(file_pair_metadata_list, file_eff_vs_purity_excl_dir, 'File Level', apa, input_file_name, all_true_metadata_list=None)
                DrawLabelPerFile(file_metadata_list, agg_output_dir, apa, input_file_name)
                DrawTrueRecoMatchMultiplicity(file_metadata_list, agg_output_dir, apa, 'File Level', 'file', file_name=input_file_name)

                #DrawAggregatedEfficiencyPlots(file_efficiency_results, agg_output_dir, "File-Level", apa)
                #DrawAggregatedPurityPlots(file_purity_results, agg_output_dir, "File-Level", apa)
            
            # ================================================================
            # FILE-LEVEL METADATA SUMMARY
            # ================================================================
            if Print_Metadata and file_metadata_list:
                file_metadata_stats = aggregate_metadata(file_metadata_list)
                #print(f"\n  FILE-LEVEL METADATA SUMMARY ({input_file_name}):")
                #print(f"  Total clusters: {file_metadata_stats['total_clusters']}")
                #print(f"  By type - Neutrino: {file_metadata_stats['by_type']['neutrino']}, Cosmic: {file_metadata_stats['by_type']['cosmic']}")
                #print(f"  By category - Isochronous: {file_metadata_stats['by_category']['isochronous']}, Prolonged: {file_metadata_stats['by_category']['prolonged']}, Normal: {file_metadata_stats['by_category']['normal']}")
                #print(f"  Efficiency - Mean: {file_metadata_stats['efficiency_stats']['mean']:.4f}, Median: {file_metadata_stats['efficiency_stats']['median']:.4f}")
                #print(f"  Reco Matches - Mean: {file_metadata_stats['reco_matches_stats']['mean']:.2f}, Median: {file_metadata_stats['reco_matches_stats']['median']:.2f}\n")

            # Aggregate to job level
            #job_efficiency_results.extend(file_efficiency_results)
            #job_purity_results.extend(file_purity_results)
            #job_matched_pairs.extend(file_matched_pairs)
            total_files_processed += 1
        
        # ========================================================================
        # JOB-LEVEL AGGREGATION (After all files are processed)
        # ========================================================================
        print(f"\n{'='*70}")
        print(f"JOB-LEVEL AGGREGATION: Generating job-level summary plots...")
        print(f"{'='*70}")
        print(f"Total files processed: {total_files_processed}")
        print(f"Total events processed: {total_events_processed}")
        print(f"Total efficiency results: {len(job_efficiency_results)}")
        print(f"Total purity results: {len(job_purity_results)}")
        print(f"Total matched pairs: {len(job_matched_pairs)}")
        print(f"Total 1-to-1 true-reco pairs: {len(job_pair_metadata_list)}")
        
        if job_efficiency_results and job_purity_results:
            job_agg_output_dir = output_dir / "job_summary"
            job_agg_output_dir.mkdir(parents=True, exist_ok=True)

            # Sub-directories for 2D/1D efficiency-vs-true-energy plots, split by matching level
            job_efficiency_2d1d_imaging_dir              = job_agg_output_dir / "efficiency" / "efficiency_2d_1d_imaging_level"
            job_efficiency_2d1d_clustering_dir           = job_agg_output_dir / "efficiency" / "efficiency_2d_1d_clusteringlevel"
            job_efficiency_2d1d_clustering_pairs_only_dir = job_agg_output_dir / "efficiency" / "efficiency_2d_1d_clusteringlevel_true_reco_pairs_only"
            job_efficiency_2d1d_imaging_dir              .mkdir(parents=True, exist_ok=True)
            job_efficiency_2d1d_clustering_dir           .mkdir(parents=True, exist_ok=True)
            job_efficiency_2d1d_clustering_pairs_only_dir.mkdir(parents=True, exist_ok=True)

            # Sub-directory for purity-vs-reco-charge plots
            job_purity_summary_dir = job_agg_output_dir / "purity"
            job_purity_summary_dir.mkdir(parents=True, exist_ok=True)

            # Sub-directories for efficiency-vs-purity matched-pair plots: one including
            # unmatched true clusters (drawn in a dedicated "no match" box), one excluding them
            job_eff_vs_purity_incl_dir = job_agg_output_dir / "true_reco_matched_pair_efficiency_purity_including_unmatched_true_clusters"
            job_eff_vs_purity_excl_dir = job_agg_output_dir / "true_reco_matched_pair_efficiency_purity_excluding_unmatched_true_clusters"
            job_eff_vs_purity_incl_dir.mkdir(parents=True, exist_ok=True)
            job_eff_vs_purity_excl_dir.mkdir(parents=True, exist_ok=True)

            imaging_energies, imaging_efficiencies = DrawEfficiencyVsTrueEnergyPerJob(job_efficiency_results, job_efficiency_2d1d_imaging_dir, apa, job_metadata_list=job_metadata_list)
            DrawClusterEfficiencyVsTrueEnergyPerJob(job_pair_metadata_list, job_efficiency_2d1d_clustering_dir, apa, all_true_metadata_list=job_metadata_list)
            clustering_pairs_energies, clustering_pairs_efficiencies = DrawEfficiencyVsTrueEnergy_MatchedPairs_PerJob(job_pair_metadata_list, job_efficiency_2d1d_clustering_pairs_only_dir, apa)
            DrawPurityVsRecoChargePerJob(job_pair_metadata_list, job_purity_summary_dir, apa)
            DrawEfficiencyVsPurity_MatchedPairs(job_pair_metadata_list, job_eff_vs_purity_incl_dir, 'Job Level', apa, all_true_metadata_list=job_metadata_list)
            DrawEfficiencyVsPurity_MatchedPairs(job_pair_metadata_list, job_eff_vs_purity_excl_dir, 'Job Level', apa, all_true_metadata_list=None)
            DrawLabelPerJob(job_metadata_list, job_agg_output_dir, apa)
            DrawTrueRecoMatchMultiplicity(job_metadata_list, job_agg_output_dir, apa, 'Job Level', 'job')

            # Phase A ROOT output for this APA - same data process_events_to_root.py would
            # produce, for later quick-turnaround analysis via Evaluation_BeforeChargeLightMatching_BeforeBeamWindowCut_FromROOT.ipynb
            # without re-running this notebook's selections/KDTree matching.
            root_out_path = job_agg_output_dir / "cluster_data.root"
            root_row_counts = write_root_file(
                root_out_path, root_true_points_cols, root_reco_points_cols, root_true_points_before_deadarea_cols,
                root_true_cluster_metadata_list, root_reco_cluster_metadata_list, root_pair_metadata_list)
            print(f"\nPhase A ROOT output written to: {root_out_path}")
            print(f"  {root_row_counts}")
            
            #DrawAggregatedEfficiencyPlots(job_efficiency_results, job_agg_output_dir, "Job-Level", apa)
            #DrawAggregatedPurityPlots(job_purity_results, job_agg_output_dir, "Job-Level", apa)
            if job_matched_pairs:
                DrawMatchedPairsPlots(job_matched_pairs, job_agg_output_dir, "Job-Level", apa)
            
            print(f"\nJob-level summary statistics:")
            print(f"  Mean Efficiency: {(np.mean(imaging_efficiencies) if imaging_efficiencies else float('nan')):.4f}")
            print(f"  Mean Purity: {np.mean([p['purity'] for p in job_purity_results]):.4f}")

            # ================================================================
            # WRITE JOB SUMMARY TO summary.txt
            # ================================================================
            energy_threshold_mev = 500.0

            def _mean_str(values):
                return f"{np.mean(values):.4f}" if values else "N/A"

            # Category counts from job-level metadata (mirrors the category breakdown
            # printed inside DrawEfficiencyVsTrueEnergyPerJob)
            neutrino_clusters_meta    = [m for m in job_metadata_list if m['cluster_type'] == 'neutrino']
            cosmic_clusters_meta      = [m for m in job_metadata_list if m['cluster_type'] == 'cosmic']
            isochronous_clusters_meta = [m for m in cosmic_clusters_meta if m['cluster_category'] == 'isochronous']
            normal_clusters_meta      = [m for m in cosmic_clusters_meta if m['cluster_category'] == 'normal']
            prolonged_clusters_meta   = [m for m in cosmic_clusters_meta if m['cluster_category'] == 'prolonged']

            # Split efficiency by true cluster energy, using the same per-true-cluster
            # (event, true_cluster_id) values plotted in efficiency_vs_true_energy_1d_job_<APA>.png
            # ("All Clusters"), not the raw (fragmented) efficiency_results pair rows.
            eff_below_500 = [eff for en, eff in zip(imaging_energies, imaging_efficiencies) if en < energy_threshold_mev]
            eff_above_500 = [eff for en, eff in zip(imaging_energies, imaging_efficiencies) if en >= energy_threshold_mev]

            # Split purity results by true cluster energy, looked up from job-level metadata
            # via (event, true_cluster_id) since purity_results don't carry true energy directly
            true_energy_lookup = {(m['event'], m['true_cluster_id']): m['total_true_energy'] for m in job_metadata_list}
            pur_below_500, pur_above_500 = [], []
            for p in job_purity_results:
                true_energy = true_energy_lookup.get((p.get('event'), p.get('true_cluster_id')))
                if true_energy is None:
                    continue
                if true_energy < energy_threshold_mev:
                    pur_below_500.append(p['purity'])
                else:
                    pur_above_500.append(p['purity'])

            # Split 1-to-1 pair (clustering-level) efficiency by true cluster energy, using
            # the same per-pair values plotted in
            # efficiency_vs_true_energy_1d_clusteringlevel_pairs_only_job_<APA>.png ("All Clusters").
            pair_eff_below_500 = [eff for en, eff in zip(clustering_pairs_energies, clustering_pairs_efficiencies)
                                   if en < energy_threshold_mev]
            pair_eff_above_500 = [eff for en, eff in zip(clustering_pairs_energies, clustering_pairs_efficiencies)
                                   if en >= energy_threshold_mev]
            pair_pur_below_500 = [m['purity'] for m in job_pair_metadata_list
                                   if m.get('total_true_energy', 0) < energy_threshold_mev]
            pair_pur_above_500 = [m['purity'] for m in job_pair_metadata_list
                                   if m.get('total_true_energy', 0) >= energy_threshold_mev]

            summary_lines = []
            summary_lines.append("=" * 80)
            summary_lines.append("JOB SUMMARY")
            summary_lines.append("=" * 80)
            summary_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            summary_lines.append("")
            summary_lines.append("Configuration:")
            summary_lines.append(f"Parent directory: {PARENT_DIR}")
            summary_lines.append(f"APA: {apa}")
            summary_lines.append(f"Plot base directory: {PLOTBASEDIR}")
            summary_lines.append(f"Time window: {time_window_min} to {time_window_max}")
            summary_lines.append(f"Files to process: {files}")
            summary_lines.append(f"Events to process: {events}")
            summary_lines.append("")
            summary_lines.append("Selection Parameters:")
            summary_lines.append(f"  radius_efficiency:        {radius_efficiency}")
            summary_lines.append(f"  radius_purity_xz:         {radius_purity_xz}")
            summary_lines.append(f"  radius_purity_yz:         {radius_purity_yz}")
            summary_lines.append(f"  radius_purity_xy:         {radius_purity_xy}")
            summary_lines.append(f"  min_recopoints_threshold: {min_recopoints_threshold}")
            summary_lines.append(f"  min_cluster_energy:       {min_cluster_energy} MeV")
            summary_lines.append(f"  min_true_points_cutoff:   {min_true_points_cutoff}")
            summary_lines.append(f"  min_reco_points_cutoff:   {min_reco_points_cutoff}")
            summary_lines.append("")
            summary_lines.append("YZ Sensitivity Cut Parameters:")
            summary_lines.append(f"  x_min: {x_min}   x_max: {x_max}")
            summary_lines.append(f"  y_min: {y_min}   y_max: {y_max}")
            summary_lines.append(f"  z_min: {z_min}   z_max: {z_max}")
            summary_lines.append("")
            summary_lines.append("Time Window Cut Values:")
            summary_lines.append(f"  time_window_min: {time_window_min} μs")
            summary_lines.append(f"  time_window_max: {time_window_max} μs")
            summary_lines.append("")
            summary_lines.append("Cuts applied:")
            if Apply_energy_cutoff:
                summary_lines.append(f"- Energy cutoff: {min_cluster_energy} MeV")
            if Apply_min_true_points_cutoff:
                summary_lines.append(f"- Minimum true points cutoff: {min_true_points_cutoff}")
            if Apply_min_reco_points_cutoff:
                summary_lines.append(f"- Minimum reco points cutoff: {min_reco_points_cutoff}")
            if Apply_wire_readout_sensitive_xz_plane_cut:
                summary_lines.append("- Wire readout sensitive xz plane cut applied")
            if Apply_time_window_cut:
                summary_lines.append(f"- Time window cut: {time_window_min} to {time_window_max} μs")
            if Apply_deadarea_cut:
                summary_lines.append("- Dead area cut applied")
            summary_lines.append("")
            summary_lines.append("Cuts not applied:")
            if not Apply_energy_cutoff:
                summary_lines.append("- Energy cutoff not applied")
            if not Apply_min_true_points_cutoff:
                summary_lines.append("- Minimum true points cutoff not applied")
            if not Apply_min_reco_points_cutoff:
                summary_lines.append("- Minimum reco points cutoff not applied")
            if not Apply_wire_readout_sensitive_xz_plane_cut:
                summary_lines.append("- Wire readout sensitive xz plane cut not applied")
            if not Apply_time_window_cut:
                summary_lines.append("- Time window cut not applied")
            if not Apply_deadarea_cut:
                summary_lines.append("- Dead area cut not applied")
            summary_lines.append("")
            summary_lines.append("=" * 80)
            summary_lines.append("JOB-LEVEL AGGREGATION")
            summary_lines.append("=" * 80)
            summary_lines.append(f"Total files processed: {total_files_processed}")
            summary_lines.append(f"Total events processed: {total_events_processed}")
            summary_lines.append(f"Total efficiency results: {len(job_efficiency_results)}")
            summary_lines.append(f"Total purity results: {len(job_purity_results)}")
            summary_lines.append(f"Total matched pairs: {len(job_matched_pairs)}")
            summary_lines.append(f"Total 1-to-1 true-reco pairs: {len(job_pair_metadata_list)}")
            summary_lines.append("")
            summary_lines.append(f"  [JOB LEVEL] Built category_info_source from metadata: {len(job_metadata_list)} clusters")
            summary_lines.append("  [JOB LEVEL] Drawing 2D efficiency plots for categories...")
            summary_lines.append(f"    Neutrino Clusters: found {len(neutrino_clusters_meta)} clusters in metadata")
            summary_lines.append(f"      → Matched {len(neutrino_clusters_meta)} clusters in efficiency_results")
            summary_lines.append(f"    Isochronous Cosmic Clusters: found {len(isochronous_clusters_meta)} clusters in metadata")
            summary_lines.append(f"      → Matched {len(isochronous_clusters_meta)} clusters in efficiency_results")
            summary_lines.append(f"    Normal Cosmic Clusters: found {len(normal_clusters_meta)} clusters in metadata")
            summary_lines.append(f"      → Matched {len(normal_clusters_meta)} clusters in efficiency_results")
            summary_lines.append(f"    Prolonged Cosmic Clusters: found {len(prolonged_clusters_meta)} clusters in metadata")
            summary_lines.append(f"      → Matched {len(prolonged_clusters_meta)} clusters in efficiency_results")
            summary_lines.append("")
            summary_lines.append("Job-level summary statistics: Imaging Level")
            summary_lines.append(f"  Mean Efficiency (overall):             {_mean_str(imaging_efficiencies)}")
            summary_lines.append(f"  Mean Efficiency (< {energy_threshold_mev:.0f} MeV):          {_mean_str(eff_below_500)}  ({len(eff_below_500)} entries)")
            summary_lines.append(f"  Mean Efficiency (>= {energy_threshold_mev:.0f} MeV):         {_mean_str(eff_above_500)}  ({len(eff_above_500)} entries)")
            summary_lines.append(f"  Mean Purity (overall):                  {_mean_str([p['purity'] for p in job_purity_results])}")
            summary_lines.append(f"  Mean Purity (< {energy_threshold_mev:.0f} MeV):               {_mean_str(pur_below_500)}  ({len(pur_below_500)} entries)")
            summary_lines.append(f"  Mean Purity (>= {energy_threshold_mev:.0f} MeV):              {_mean_str(pur_above_500)}  ({len(pur_above_500)} entries)")
            summary_lines.append("=" * 80)
            summary_lines.append("")
            summary_lines.append("Job-level summary statistics: Clustering Level")
            summary_lines.append(f"  Mean Efficiency (overall):             {_mean_str(clustering_pairs_efficiencies)}")
            summary_lines.append(f"  Mean Efficiency (< {energy_threshold_mev:.0f} MeV):          {_mean_str(pair_eff_below_500)}  ({len(pair_eff_below_500)} entries)")
            summary_lines.append(f"  Mean Efficiency (>= {energy_threshold_mev:.0f} MeV):         {_mean_str(pair_eff_above_500)}  ({len(pair_eff_above_500)} entries)")
            summary_lines.append(f"  Mean Purity (overall):                  {_mean_str([m['purity'] for m in job_pair_metadata_list])}")
            summary_lines.append(f"  Mean Purity (< {energy_threshold_mev:.0f} MeV):               {_mean_str(pair_pur_below_500)}  ({len(pair_pur_below_500)} entries)")
            summary_lines.append(f"  Mean Purity (>= {energy_threshold_mev:.0f} MeV):              {_mean_str(pair_pur_above_500)}  ({len(pair_pur_above_500)} entries)")
            summary_lines.append("=" * 80)

            summary_file = job_agg_output_dir / "summary.txt"
            with open(summary_file, 'w') as f:
                f.write('\n'.join(summary_lines) + '\n')
            print(f"\nJob summary saved to: {summary_file}")
        
        # ================================================================
        # JOB-LEVEL METADATA SUMMARY
        # ================================================================
        if job_metadata_list:
            job_metadata_stats = aggregate_metadata(job_metadata_list)

            Print_Metadata = False  # Set to True to print job-level metadata summary

            if Print_Metadata:
                print(f"\n{'='*70}")
                print(f"JOB-LEVEL METADATA SUMMARY:")
                print(f"{'='*70}")
                print(f"Total clusters across all files: {job_metadata_stats['total_clusters']}")
                print(f"\nBy type:")
                print(f"  Neutrino: {job_metadata_stats['by_type']['neutrino']}")
                print(f"  Cosmic: {job_metadata_stats['by_type']['cosmic']}")
                print(f"\nBy category:")
                print(f"  Isochronous: {job_metadata_stats['by_category']['isochronous']}")
                print(f"  Prolonged: {job_metadata_stats['by_category']['prolonged']}")
                print(f"  Normal: {job_metadata_stats['by_category']['normal']}")
                print(f"\nEfficiency Statistics:")
                print(f"  Mean: {job_metadata_stats['efficiency_stats']['mean']:.4f}")
                print(f"  Median: {job_metadata_stats['efficiency_stats']['median']:.4f}")
                print(f"  Min: {job_metadata_stats['efficiency_stats']['min']:.4f}")
                print(f"  Max: {job_metadata_stats['efficiency_stats']['max']:.4f}")
                print(f"\nReco Matches Statistics:")
                print(f"  Mean: {job_metadata_stats['reco_matches_stats']['mean']:.2f}")
                print(f"  Median: {job_metadata_stats['reco_matches_stats']['median']:.2f}")
                print(f"  Min: {job_metadata_stats['reco_matches_stats']['min']}")
                print(f"  Max: {job_metadata_stats['reco_matches_stats']['max']}")
                print(f"{'='*70}\n")
        
        # ========================================================================
        # SAVE BEE DISPLAY LINKS TO FILE
        # ========================================================================
        if job_bee_links:
            bee_links_file = output_dir / "job_summary" / "bee_display_links.txt"
            with open(bee_links_file, 'w') as f:
                f.write("BEE DISPLAY LINKS FOR ALL FILES\n")
                f.write("=" * 80 + "\n\n")
                for link_info in job_bee_links:
                    f.write(f"{link_info['file']}:\n")
                    f.write(f"  {link_info['url']}\n\n")
            print(f"\nBee display links saved to: {bee_links_file}")
        else:
            print("\nNo bee display links to save")

        # ========================================================================
        # JOB RUNTIME
        # ========================================================================
        job_end_time        = time.time()
        job_elapsed_seconds = job_end_time - job_start_time
        job_elapsed_str      = str(timedelta(seconds=int(job_elapsed_seconds)))
        print(f"\n{'='*70}")
        print(f"Job finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Total job runtime: {job_elapsed_str} ({job_elapsed_seconds:.1f} seconds)")
        print(f"{'='*70}")

        if job_efficiency_results and job_purity_results:
            with open(job_agg_output_dir / "summary.txt", 'a') as f:
                f.write("\n")
                f.write("=" * 80 + "\n")
                f.write("JOB RUNTIME\n")
                f.write("=" * 80 + "\n")
                f.write(f"Job started at: {datetime.fromtimestamp(job_start_time).strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"Job finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"Total job runtime: {job_elapsed_str} ({job_elapsed_seconds:.1f} seconds)\n")
            print(f"Job runtime appended to: {job_agg_output_dir / 'summary.txt'}")


######################################################################
PROCESSING APA0
######################################################################


📁 Output directory created with timestamp:
   multi_file_plots_with_deghosting/2view/apa_APA0_20260725_190737


FILE 1/10: 2view (2view/file1)

Processing events 0 to 7

FILE 1/10: 2view (2view/file1)
  EVENT 0
Applying Dead Area Cut
APA: APA0, View: 2view
Number of points before dead area cut: 39710
Number of points after dead area cut: 39710
Points removed: 0 (0.0%)

Drew before/after dead area visualizations
    Analyzing cluster categories for 4 true clusters...

    Cluster category analysis complete: 4 clusters analyzed
Found 4 true clusters with matched reco clusters (1-to-many)
Found 4 matched pairs of true and reco clusters (1-to-1)
    Metadata collected for 4 clusters
Found 4 matched pairs of true and reco clusters (1-to-1)
    Pair metadata collected for 4 true-reco pairs
    [1/5] Drawing efficiency/purity heatmaps

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/efficiency_purity_draw.py:87: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  plt.hist2d(energies, efficiencies, bins=[xbins, ybins], cmap='YlGnBu')
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/efficiency_purity_draw.py:102: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  plt.xlim(0, max(energies)*1.1)
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/efficiency_purity_draw.py:491: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(fontsize=10)
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/efficiency_purity_draw.py:529: UserWarning: No artists with labels found to put in legend.  Note that artists whose label star

    [4/5] Drawing purity vs reco charge plots...
FILE 1/10: 2view (2view/file1)
  EVENT 0
FILE 1/10: 2view (2view/file1)
  EVENT 0
    [5/5] Event 0 complete: 4 true, 5 reco clusters

FILE 1/10: 2view (2view/file1)
  EVENT 1
Applying Dead Area Cut
APA: APA0, View: 2view
Number of points before dead area cut: 10115
Number of points after dead area cut: 10115
Points removed: 0 (0.0%)

Drew before/after dead area visualizations
    Analyzing cluster categories for 3 true clusters...

    Cluster category analysis complete: 3 clusters analyzed
Found 3 true clusters with matched reco clusters (1-to-many)
Found 3 matched pairs of true and reco clusters (1-to-1)
    Metadata collected for 3 clusters
Found 3 matched pairs of true and reco clusters (1-to-1)
    Pair metadata collected for 3 true-reco pairs
    [1/5] Drawing efficiency/purity heatmaps...
FILE 1/10: 2view (2view/file1)
  EVENT 1
    [2/5] Drawing true clusters with matched reco clusters...
FILE 1/10: 2view (2view/file1)
  EVENT 1

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/efficiency_purity_draw.py:87: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  plt.hist2d(energies, efficiencies, bins=[xbins, ybins], cmap='YlGnBu')
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/efficiency_purity_draw.py:102: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  plt.xlim(0, max(energies)*1.1)
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/efficiency_purity_draw.py:491: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(fontsize=10)
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/efficiency_purity_draw.py:529: UserWarning: No artists with labels found to put in legend.  Note that artists whose label star

    [4/5] Drawing purity vs reco charge plots...
FILE 1/10: 2view (2view/file1)
  EVENT 1
FILE 1/10: 2view (2view/file1)
  EVENT 1
    [5/5] Event 1 complete: 3 true, 4 reco clusters

FILE 1/10: 2view (2view/file1)
  EVENT 2
Applying Dead Area Cut
APA: APA0, View: 2view
Number of points before dead area cut: 68992
Number of points after dead area cut: 68971
Points removed: 21 (0.0%)
  Cluster 9999: 18472 -> 18451 points (21 removed, 0.1%)

Drew before/after dead area visualizations

Drawing 1 clusters affected by dead area...


...

    [4/5] Drawing purity vs reco charge plots...
FILE 10/10: 2view (2view/file9)
  EVENT 9
FILE 10/10: 2view (2view/file9)
  EVENT 9
    [5/5] Event 9 complete: 4 true, 5 reco clusters

FILE 10/10: 2view (2view/file9)
  EVENT 10
Applying Dead Area Cut
APA: APA0, View: 2view
Number of points before dead area cut: 36014
Number of points after dead area cut: 36014
Points removed: 0 (0.0%)

Drew before/after dead area visualizations
    Analyzing cluster categories for 5 true clusters...

    Cluster category analysis complete: 5 clusters analyzed
Found 5 true clusters with matched reco clusters (1-to-many)
Found 5 matched pairs of true and reco clusters (1-to-1)
    Metadata collected for 5 clusters
Found 5 matched pairs of true and reco clusters (1-to-1)
    Pair metadata collected for 5 true-reco pairs
    [1/5] Drawing efficiency/purity heatmaps...
FILE 10/10: 2view (2view/file9)
  EVENT 10
    [2/5] Drawing true clusters with matched reco clusters...
FILE 10/10: 2view (2view/file9)
  

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/efficiency_purity_draw.py:87: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  plt.hist2d(energies, efficiencies, bins=[xbins, ybins], cmap='YlGnBu')
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/efficiency_purity_draw.py:102: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  plt.xlim(0, max(energies)*1.1)
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/efficiency_purity_draw.py:491: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(fontsize=10)
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/efficiency_purity_draw.py:529: UserWarning: No artists with labels found to put in legend.  Note that artists whose label star

    [4/5] Drawing purity vs reco charge plots...
FILE 10/10: 2view (2view/file9)
  EVENT 10
FILE 10/10: 2view (2view/file9)
  EVENT 10
    [5/5] Event 10 complete: 5 true, 6 reco clusters


FILE-LEVEL AGGREGATION: Generating file-level summary plots...
FILE 10/10: 2view (2view/file9)
Total events processed in file: 118
Total efficiency results in file: 61
Total purity results in file: 70
Total matched pairs in file: 0
Total 1-to-1 true-reco pairs in file: 56

  FILE-LEVEL AGGREGATION: Generating file-level summary plots...
  Total clusters in file: 61 efficiency, 70 purity
    [FILE LEVEL] Built category_info_source from metadata: 56 clusters
    [FILE LEVEL] Drawing 2D efficiency plots for categories...
      Neutrino Clusters: found 1 clusters in metadata
        → Matched 1 clusters in efficiency_results
        → Drawing 2D plot for Neutrino Clusters...
      Isochronous Cosmic Clusters: found 9 clusters in metadata
        → Matched 9 clusters in efficiency_results
        → Drawi

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/process_events_to_root.py:384: FutureWarning: Starting in version 5.7.0, Uproot will default to writing RNTuples instead of TTrees. You will need to use `mktree` to explicitly create a TTree. This can be done by changing `file['tree_name'] = data` to `file.mktree('tree_name', data)`. Please update your code accordingly.
  f["true_points"] = true_points_data



Phase A ROOT output written to: multi_file_plots_with_deghosting/2view/apa_APA0_20260725_190737/job_summary/cluster_data.root
  {'true_points': 4916453, 'reco_points': 1980620, 'true_points_before_deadarea': 448927, 'true_cluster_metadata': 622, 'reco_cluster_metadata': 699, 'true_reco_pair_metadata': 620}

Job-level summary statistics:
  Mean Efficiency: 0.9581
  Mean Purity: 0.7194

Job summary saved to: multi_file_plots_with_deghosting/2view/apa_APA0_20260725_190737/job_summary/summary.txt

No bee display links to save

Job finished at: 2026-07-25 19:46:06
Total job runtime: 0:38:29 (2310.0 seconds)
Job runtime appended to: multi_file_plots_with_deghosting/2view/apa_APA0_20260725_190737/job_summary/summary.txt
